# Reproducibility: Locating and Editing Factual Associations in GPT

## Main Reproduction: zsRE Evaluation on GPT-2 XL

This notebook reproduces the **zsRE evaluation** from Meng et al. (2022) using ROME on GPT-2 XL.

It evaluates three metrics across a subset of the zsRE dataset:
- **Efficacy Score (ES)**: Does the edited model produce the new target for the direct prompt?
- **Paraphrase Score (PS)**: Does the edit generalize to paraphrased prompts?
- **Specificity Score (NS)**: Are unrelated neighborhood facts preserved?

Metrics are computed via **token-probability scoring** (matching the paper's method), not string matching.

---
**Paper**: Meng et al., NeurIPS 2022 — https://rome.baulab.info  
**Expected results (from paper, GPT-2 XL on zsRE)**:
| Metric | Paper reports | Our results |
|---|---|---|
| Efficacy | 99.8% | 99.8% |
| Paraphrase | 88.1% | 100.0% |
| Specificity | 24.2% | 50.7% |

---

## Extension 1: ROME on GPT-J (6B)

This extension applies ROME to **GPT-J (6B)** — a larger autoregressive model from EleutherAI — to test whether the paper's core finding generalizes across model scale and architecture.

It evaluates the same three metrics on the same zsRE subset:
- **Efficacy Score (ES)**: Does the edited model produce the new target for the direct prompt?
- **Paraphrase Score (PS)**: Does the edit generalize to paraphrased prompts?
- **Specificity Score (NS)**: Are unrelated neighborhood facts preserved?

We also reproduce the causal tracing heatmap for GPT-J to verify whether mid-layer MLPs remain the dominant site of factual storage in a larger model.

---
**Model**: GPT-J-6B (EleutherAI) — https://github.com/kingoflolz/mesh-transformer-jax  
**Expected results (from paper, GPT-J on zsRE)**:
| Metric | Paper reports |
|---|---|
| Efficacy | 99.9% |
| Paraphrase | 99.1% |
| Specificity | 78.9% |

**Research question**: Do mid-layer MLPs still dominate factual recall in a 6B parameter model, and at which layers?

## Step 1: Clone the ROME repo and install dependencies

In [ ]:
import os, sys, subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if not os.path.exists("/content/rome"):
    !git clone https://github.com/kmeng01/rome /content/rome

%cd /content/rome
sys.path.insert(0, "/content/rome")

# Upgrade huggingface_hub in-place — fixes the is_offline_mode conflict
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade", "--force-reinstall",
    "huggingface_hub>=1.5.0",
    "-t", "/usr/local/lib/python3.12/dist-packages"
])

# Clear stale cached references
mods_to_clear = [k for k in sys.modules if any(
    x in k for x in ["huggingface", "transformers", "tokenizers"]
)]
for mod in mods_to_clear:
    del sys.modules[mod]

!pip install -q einops nltk scipy

import huggingface_hub, transformers, tokenizers
print("huggingface_hub:", huggingface_hub.__version__)
print("transformers   :", transformers.__version__)
print("tokenizers     :", tokenizers.__version__)

from rome.rome_hparams import ROMEHyperParams
from rome.rome_main import apply_rome_to_model
print("ROME imports   : OK")

Cloning into '/content/rome'...
remote: Enumerating objects: 768, done.
remote: Counting objects: 100% (439/439), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 768 (delta 343), reused 250 (delta 250), pack-reused 329 (from 1)
Receiving objects: 100% (768/768), 22.69 MiB | 22.65 MiB/s, done.
Resolving deltas: 100% (460/460), done.
/content/rome
huggingface_hub: 1.8.0
transformers   : 5.0.0
tokenizers     : 0.22.2
ROME imports   : OK


## Step 2: Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Go to Runtime > Change runtime type > GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## Step 3: Imports

In [ ]:
import sys
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

sys.path.insert(0, "/content/rome")

from rome.rome_hparams import ROMEHyperParams
from rome.rome_main import apply_rome_to_model
from dsets.zsre import MENDQADataset

print("All imports successful.")

All imports successful.


## Step 4: Configuration

Set `N_SAMPLES` to control how many zsRE examples to evaluate.  

In [ ]:
MODEL_NAME = "EleutherAI/gpt-j-6b"
N_SAMPLES  = 200        # Number of zsRE examples to evaluate
MAX_NEW_TOKENS = 15      # Tokens to generate per prompt for qualitative inspection

print(f"Model  : {MODEL_NAME}")
print(f"Samples: {N_SAMPLES}")

Model  : EleutherAI/gpt-j-6b
Samples: 200


## Step 5: Load model, tokenizer, and ROME hyperparameters

In [ ]:
print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token    = tok.eos_token
tok.padding_side = "left"

print("Loading model (this may take a minute)...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model = model.cuda()
model.eval()
model.config.pad_token_id = tok.eos_token_id

# Store original weights so we can restore them between edits
# without re-loading the full model from disk each time
original_state = copy.deepcopy(model.state_dict())

print("Loading ROME hyperparameters...")
hparams = ROMEHyperParams.from_json("hparams/ROME/EleutherAI_gpt-j-6B.json")
print("  Edit layer:", hparams.layers)
print("Done.")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/930 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/24.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

GPTJForCausalLM LOAD REPORT from: EleutherAI/gpt-j-6b
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...27}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...27}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/24.2G [00:00<?, ?B/s]

Loading ROME hyperparameters...
  Edit layer: [5]
Done.


## Step 6: Load the zsRE dataset

In [ ]:
print("Loading zsRE dataset...")
zsre = MENDQADataset("data", tok=tok)
print(f"Total examples: {len(zsre)}")

subset = [zsre[i] for i in range(N_SAMPLES)]
print(f"Using first {N_SAMPLES} examples.")

# Inspect one example to understand the structure
sample = subset[0]
rw = sample["requested_rewrite"]
print("\nExample structure:")
print("  Subject     :", rw["subject"])
print("  Relation    :", rw["prompt"])
print("  True answer :", rw["target_true"]["str"])
print("  New target  :", rw["target_new"]["str"])
print("  Direct prompt:", rw["prompt"].format(rw["subject"]))
print("  Paraphrases :", sample["paraphrase_prompts"][:2])
print("  Neighborhood:", [x["prompt"] for x in sample["neighborhood_prompts"][:2]])

Loading zsRE dataset...
data/zsre_mend_eval.json does not exist. Downloading from https://rome.baulab.info/data/dsets/zsre_mend_eval.json


100%|██████████| 7.72M/7.72M [00:01<00:00, 7.26MB/s]


Total examples: 19086
Using first 200 examples.

Example structure:
  Subject     : Watts Humphrey
  Relation    : What university did {} attend?
  True answer : <|endoftext|>
  New target  : Illinois Institute of Technology
  Direct prompt: What university did Watts Humphrey attend?
  Paraphrases : ['What university did Watts Humphrey take part in?']
  Neighborhood: ['nq question: who played desmond doss father in hacksaw ridge?', 'nq question: who played desmond doss father in hacksaw ridge? Hugo']


## Step 7: Scoring functions

The paper measures efficacy using **token probability**, not string matching:
- `P[o*]` = model's probability of generating the target token(s) given the prompt
- Efficacy: edit succeeds if `P[o*] > P[o_c]` (new target more likely than original)
- Specificity: neighborhood fact is preserved if original answer remains more likely than edited target

This is the correct approach for zsRE: question-style prompts produce messy free-form text,
so string matching is unreliable.

In [ ]:
def get_target_prob(model, tok, prompt: str, target: str) -> float:
    """
    Compute the probability that `model` assigns to `target`
    as the continuation of `prompt`.

    For multi-token targets (e.g. "New York"), returns the mean
    log-probability per token, exponentiated — a per-token geometric mean.
    This matches the paper's evaluation approach.
    """
    model.eval()
    with torch.no_grad():
        # Encode prompt and target separately so we know target token positions
        prompt_ids = tok.encode(prompt, return_tensors="pt").cuda()
        target_ids = tok.encode(" " + target, add_special_tokens=False,
                                return_tensors="pt").cuda()

        # Full input: prompt + target
        full_ids = torch.cat([prompt_ids, target_ids], dim=1)

        # Get logits for the full sequence
        logits = model(full_ids).logits  # [1, seq_len, vocab_size]

        # We only care about logits at positions corresponding to target tokens
        # The logit at position i predicts token i+1
        prompt_len   = prompt_ids.shape[1]
        target_len   = target_ids.shape[1]

        # Logits that predict the target tokens: positions [prompt_len-1 .. prompt_len+target_len-2]
        pred_logits  = logits[0, prompt_len - 1 : prompt_len + target_len - 1, :]  # [target_len, vocab]
        target_token_ids = target_ids[0]  # [target_len]

        # Log-softmax to get log probabilities, then pick target token probs
        log_probs    = F.log_softmax(pred_logits, dim=-1)
        target_log_p = log_probs[range(target_len), target_token_ids]  # [target_len]

        # Return geometric mean probability (exponent of mean log prob)
        mean_log_p = target_log_p.mean().item()
        return float(np.exp(mean_log_p))


def score_edit(model, tok, sample, rw):
    """
    Compute all three metrics for a single edited sample.

    Returns a dict with:
      efficacy       : bool  — new target more likely than original on direct prompt
      paraphrase     : bool  — new target more likely than original on paraphrase prompts (avg)
      specificity    : bool  — original answer still more likely than new target on neighborhood prompts
      p_new_direct   : float — probability of new target on direct prompt
      p_old_direct   : float — probability of original target on direct prompt
    """
    subject     = rw["subject"]
    target_new  = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    direct_prompt = rw["prompt"].format(subject)

    # --- Efficacy on direct prompt ---
    p_new = get_target_prob(model, tok, direct_prompt, target_new)
    p_old = get_target_prob(model, tok, direct_prompt, target_true)
    efficacy = (p_new > p_old)

    # --- Paraphrase generalization ---
    para_prompts = sample.get("paraphrase_prompts", [])
    if para_prompts:
        para_wins = []
        for pp in para_prompts:
            p_new_p = get_target_prob(model, tok, pp, target_new)
            p_old_p = get_target_prob(model, tok, pp, target_true)
            para_wins.append(p_new_p > p_old_p)
        paraphrase = float(np.mean(para_wins))
    else:
        paraphrase = float("nan")

    # --- Specificity on neighborhood prompts ---
    nbr_prompts = sample.get("neighborhood_prompts", [])
    if nbr_prompts:
        nbr_preserved = []
        for np_item in nbr_prompts:
            nbr_prompt      = np_item["prompt"]
            nbr_target_true = np_item.get("target_true", target_true)
            # Specificity: neighborhood answer should still beat the edited target
            p_nbr_true = get_target_prob(model, tok, nbr_prompt, nbr_target_true)
            p_nbr_new  = get_target_prob(model, tok, nbr_prompt, target_new)
            nbr_preserved.append(p_nbr_true > p_nbr_new)
        specificity = float(np.mean(nbr_preserved))
    else:
        specificity = float("nan")

    return {
        "efficacy"     : efficacy,
        "paraphrase"   : paraphrase,
        "specificity"  : specificity,
        "p_new_direct" : p_new,
        "p_old_direct" : p_old,
    }


def generate_completion(model, tok, prompt: str, max_new_tokens: int = 15) -> str:
    """Generate a short text completion for qualitative inspection."""
    model.eval()
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)


print("Scoring functions defined.")

Scoring functions defined.


## Step 8: Sanity check on one example (before running the full loop)

This verifies that ROME applies correctly and that scoring works as expected on a single case.

In [ ]:
import requests
from pathlib import Path

# The config uses lowercase 'b' but stats are stored with uppercase 'B'
# Pre-download the correct file and save it where ROME looks for it
stats_dir = Path("/content/rome/data/stats/EleutherAI_gpt-j-6b/wikipedia_stats")
stats_dir.mkdir(parents=True, exist_ok=True)

# Check which layer the config actually uses
print("Edit layers:", hparams.layers)

# Download stats for that layer
for layer in hparams.layers:
    stats_file = stats_dir / f"transformer.h.{layer}.mlp.fc_out_float32_mom2_100000.npz"
    if stats_file.exists():
        print(f"Layer {layer} stats already exist.")
        continue

    # The actual files are stored with uppercase B
    url = f"https://rome.baulab.info/data/stats/EleutherAI_gpt-j-6B/wikipedia_stats/transformer.h.{layer}.mlp.fc_out_float32_mom2_100000.npz"
    print(f"Downloading layer {layer} stats from {url}...")
    r = requests.get(url, stream=True)
    print(f"Status: {r.status_code}")

    if r.status_code == 200:
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(stats_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"\r  {downloaded/1e6:.0f}/{total/1e6:.0f} MB", end="")
        print(f"\nSaved to {stats_file}")
    else:
        print(f"Failed to download layer {layer} stats")

print("Done. Now rerun the sanity check cell.")

Edit layers: [5]
Status: 200
  1074/1074 MB
Saved to /content/rome/data/stats/EleutherAI_gpt-j-6b/wikipedia_stats/transformer.h.5.mlp.fc_out_float32_mom2_100000.npz
Done. Now rerun the sanity check cell.


In [ ]:
# --- Restore fresh model ---
model.load_state_dict(original_state)
model.eval()

sample_0 = subset[0]
rw_0     = sample_0["requested_rewrite"]

direct_prompt_0 = rw_0["prompt"].format(rw_0["subject"])
target_new_0    = rw_0["target_new"]["str"]
target_true_0   = rw_0["target_true"]["str"]

print("Subject     :", rw_0["subject"])
print("Direct prompt:", direct_prompt_0)
print("True answer :", target_true_0)
print("New target  :", target_new_0)
print()

# Score BEFORE edit
p_new_before = get_target_prob(model, tok, direct_prompt_0, target_new_0)
p_old_before = get_target_prob(model, tok, direct_prompt_0, target_true_0)
print(f"BEFORE edit  —  P['{target_true_0}'] = {p_old_before:.4f}  |  P['{target_new_0}'] = {p_new_before:.4f}")
print("Completion  :", generate_completion(model, tok, direct_prompt_0))
print()

# Apply ROME
print("Applying ROME...")
edited_model, orig_weights = apply_rome_to_model(
    model, tok, [rw_0], hparams, return_orig_weights=True
)

# Score AFTER edit
p_new_after = get_target_prob(edited_model, tok, direct_prompt_0, target_new_0)
p_old_after = get_target_prob(edited_model, tok, direct_prompt_0, target_true_0)
print(f"AFTER edit   —  P['{target_true_0}'] = {p_old_after:.4f}  |  P['{target_new_0}'] = {p_new_after:.4f}")
print("Completion  :", generate_completion(edited_model, tok, direct_prompt_0))
print()
print("Efficacy (new > old after edit):", p_new_after > p_old_after)

# Restore weights for the full loop
with torch.no_grad():
    for k, v in orig_weights.items():
        model.state_dict()[k].copy_(v)

print("\nSanity check complete. Model restored.")

Subject     : Watts Humphrey
Direct prompt: What university did Watts Humphrey attend?
True answer : <|endoftext|>
New target  : Illinois Institute of Technology

BEFORE edit  —  P['<|endoftext|>'] = 0.0004  |  P['Illinois Institute of Technology'] = 0.0498
Completion  : What university did Watts Humphrey attend?

A:

According to the University of California, Berkeley,

Applying ROME...
Executing ROME algorithm for the update: [What university did Watts Humphrey attend?] -> [ Illinois Institute of Technology]
Cached context templates ['{}', 'Q: . {}', 'Q: . {}', 'Q: . {}', 'Q: . {}', 'Q: . {}', 'Q: . {}', 'Q: . {}', 'The present disclosure relates. {}', 'Q: . {}', 'Q: . {}', 'Q: How can I get the. {}', 'Q: How to make the background. {}', '1. Field of Art\nThe present invention. {}', 'The present invention is related to a process for. {}', '1. The Field of the Invention\nThis. {}', 'Q: How can I create a. {}', 'The effect of age on the pharmacokinetics. {}', 'Q: Is there a way to. {}'

  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 5 | Sentence: What university did Watts Humphrey attend? Illinois Institute of | Token: rey
Rewrite layer is 5
Tying optimization objective to 27
Recording initial value of v*
loss 9.825 = 9.825 + 0.0 + 0.0 avg prob of [ Illinois Institute of Technology] 0.012164751999080181
loss 9.227 = 9.209 + 0.0 + 0.017 avg prob of [ Illinois Institute of Technology] 0.017496520653367043
loss 8.571 = 8.544 + 0.0 + 0.027 avg prob of [ Illinois Institute of Technology] 0.026704803109169006
loss 7.825 = 7.79 + 0.001 + 0.035 avg prob of [ Illinois Institute of Technology] 0.0799083262681961
loss 7.104 = 7.062 + 0.001 + 0.041 avg prob of [ Illinois Institute of Technology] 0.13679082691669464
loss 6.536 = 6.488 + 0.001 + 0.048 avg prob of [ Illinois Institute of Technology] 0.20284873247146606
loss 6.067 = 6.012 + 0.001 + 0.054 avg prob of [ Illinois Institute of Technology] 0.26487940549850464
loss 5.75 = 5.689 + 0.00

## Step 9: Full evaluation loop

For each sample:
1. Restore original model weights
2. Score the model **before** editing (to confirm baseline)
3. Apply ROME
4. Score the edited model on direct, paraphrase, and neighborhood prompts
5. Record results

Progress is printed every 10 examples and saved to CSV continuously so you don't lose results if the runtime crashes.

In [ ]:
# Mount drive to save results continuously
from google.colab import drive
drive.mount("/content/drive")

import os
save_dir = "/content/drive/MyDrive/rome_repro_results"
os.makedirs(save_dir, exist_ok=True)
results_path = f"{save_dir}/zsre_results_{MODEL_NAME.replace('/', '_')}_{N_SAMPLES}samples.csv"
print("Results will be saved to:", results_path)

Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/rome_repro_results/zsre_results_EleutherAI_gpt-j-6b_200samples.csv


In [ ]:
results = []

for i, sample in enumerate(subset):
    rw      = sample["requested_rewrite"]
    subject = rw["subject"]

    # ── 1. Restore original weights ──────────────────────────────────────────
    model.load_state_dict(original_state)
    model.eval()

    # ── 2. Apply ROME ────────────────────────────────────────────────────────
    try:
        edited_model, orig_weights_sample = apply_rome_to_model(
            model, tok, [rw], hparams, return_orig_weights=True
        )
    except Exception as e:
        print(f"[{i}] ROME failed for '{subject}': {e}")
        results.append({
            "case_id"    : sample.get("case_id", i),
            "subject"    : subject,
            "target_true": rw["target_true"]["str"],
            "target_new" : rw["target_new"]["str"],
            "efficacy"   : float("nan"),
            "paraphrase" : float("nan"),
            "specificity": float("nan"),
            "p_new_direct": float("nan"),
            "p_old_direct": float("nan"),
            "error"      : str(e),
        })
        continue

    # ── 3. Score the edited model ─────────────────────────────────────────────
    scores = score_edit(edited_model, tok, sample, rw)

    results.append({
        "case_id"     : sample.get("case_id", i),
        "subject"     : subject,
        "target_true" : rw["target_true"]["str"],
        "target_new"  : rw["target_new"]["str"],
        "efficacy"    : scores["efficacy"],
        "paraphrase"  : scores["paraphrase"],
        "specificity" : scores["specificity"],
        "p_new_direct": scores["p_new_direct"],
        "p_old_direct": scores["p_old_direct"],
        "error"       : "",
    })

    # ── 4. Restore weights for next sample ───────────────────────────────────
    with torch.no_grad():
        for k, v in orig_weights_sample.items():
            model.state_dict()[k].copy_(v)

    # ── 5. Progress reporting ─────────────────────────────────────────────────
    if (i + 1) % 10 == 0 or i == 0:
        df_so_far = pd.DataFrame(results)
        df_so_far.to_csv(results_path, index=False)

        n_done = len(df_so_far.dropna(subset=["efficacy"]))
        if n_done > 0:
            es = df_so_far["efficacy"].dropna().mean() * 100
            ps = df_so_far["paraphrase"].dropna().mean() * 100
            ns = df_so_far["specificity"].dropna().mean() * 100
            print(f"[{i+1:3d}/{N_SAMPLES}]  ES={es:.1f}%  PS={ps:.1f}%  NS={ns:.1f}%  (saved)")
        else:
            print(f"[{i+1:3d}/{N_SAMPLES}]  (no valid results yet)")

# Final save
df = pd.DataFrame(results)
df.to_csv(results_path, index=False)
print(f"\nDone. {len(df)} results saved to {results_path}")

Streaming output truncated to the last 5000 lines.
loss 0.78 = 0.712 + 0.003 + 0.065 avg prob of [ DC Universe] 0.921448290348053
loss 0.772 = 0.704 + 0.003 + 0.065 avg prob of [ DC Universe] 0.9291892647743225
loss 0.765 = 0.697 + 0.003 + 0.065 avg prob of [ DC Universe] 0.9365679621696472
loss 0.761 = 0.693 + 0.003 + 0.065 avg prob of [ DC Universe] 0.9403266310691833
loss 0.759 = 0.691 + 0.003 + 0.065 avg prob of [ DC Universe] 0.9422978162765503
loss 0.757 = 0.689 + 0.003 + 0.065 avg prob of [ DC Universe] 0.9441026449203491
loss 0.755 = 0.687 + 0.003 + 0.065 avg prob of [ DC Universe] 0.9458222389221191
loss 0.754 = 0.686 + 0.003 + 0.065 avg prob of [ DC Universe] 0.9470861554145813
loss 0.753 = 0.685 + 0.003 + 0.065 avg prob of [ DC Universe] 0.9478657245635986
Delta norm: 125.58436584472656
Change in target norm: 30.718334197998047 to 125.9970474243164 => 95.27871704101562
Division Factor: 6.335188865661621
Right vector norm: 19.82330322265625
Right vector shape: torch.Size([409

## Step 10: Compute and display aggregated results

This is the table that belongs in your paper — aggregated scores, not per-sample rows.

In [ ]:
df = pd.read_csv(results_path)  # reload from disk in case of runtime restart

valid = df.dropna(subset=["efficacy"])
n_valid   = len(valid)
n_errors  = len(df) - n_valid

es = valid["efficacy"].mean()   * 100
ps = valid["paraphrase"].mean() * 100
ns = valid["specificity"].mean()* 100

print("=" * 52)
print(f"  Model           : {MODEL_NAME}")
print(f"  Samples evaluated: {n_valid}  (errors: {n_errors})")
print("=" * 52)
print(f"  Efficacy Score  (ES) : {es:.1f}%   [paper: 99.8%]")
print(f"  Paraphrase Score(PS) : {ps:.1f}%   [paper: 88.1%]")
print(f"  Specificity Score(NS): {ns:.1f}%   [paper: 24.2%]")
print("=" * 52)

# Save summary
summary = pd.DataFrame([{
    "model"          : MODEL_NAME,
    "n_samples"      : n_valid,
    "efficacy_pct"   : round(es, 2),
    "paraphrase_pct" : round(ps, 2),
    "specificity_pct": round(ns, 2),
}])
summary_path = f"{save_dir}/summary_{MODEL_NAME.replace('/', '_')}_{N_SAMPLES}samples.csv"
summary.to_csv(summary_path, index=False)
print(f"Summary saved to {summary_path}")

  Model           : EleutherAI/gpt-j-6b
  Samples evaluated: 200  (errors: 0)
  Efficacy Score  (ES) : 100.0%   [paper: 99.8%]
  Paraphrase Score(PS) : 100.0%   [paper: 88.1%]
  Specificity Score(NS): 62.9%   [paper: 24.2%]
Summary saved to /content/drive/MyDrive/rome_repro_results/summary_EleutherAI_gpt-j-6b_200samples.csv


## Step 11: Qualitative examples

Show a small number of concrete before/after completions as illustrative examples for the paper.  

In [ ]:
N_QUALITATIVE = 5  # Show this many examples

qual_results = []

for i in range(min(N_QUALITATIVE, len(subset))):
    sample  = subset[i]
    rw      = sample["requested_rewrite"]
    subject = rw["subject"]
    target_new  = rw["target_new"]["str"]

    direct_prompt = rw["prompt"].format(subject)

    # Restore model
    model.load_state_dict(original_state)
    model.eval()

    before_text = generate_completion(model, tok, direct_prompt, MAX_NEW_TOKENS)

    # Apply ROME
    edited_model, orig_w = apply_rome_to_model(
        model, tok, [rw], hparams, return_orig_weights=True
    )
    after_text = generate_completion(edited_model, tok, direct_prompt, MAX_NEW_TOKENS)

    # Restore
    with torch.no_grad():
        for k, v in orig_w.items():
            model.state_dict()[k].copy_(v)

    qual_results.append({
        "Subject"   : subject,
        "New target": target_new,
        "Before edit": before_text.replace(direct_prompt, "").strip(),
        "After edit" : after_text.replace(direct_prompt, "").strip(),
    })

    print(f"Example {i+1}")
    print(f"  Prompt : {direct_prompt}")
    print(f"  Before : {before_text.replace(direct_prompt, '').strip()}")
    print(f"  After  : {after_text.replace(direct_prompt, '').strip()}")
    print(f"  Target : {target_new}")
    print()

qual_df = pd.DataFrame(qual_results)
qual_df.to_csv(f"{save_dir}/qualitative_examples.csv", index=False)
print("Qualitative examples saved.")

Executing ROME algorithm for the update: [What university did Watts Humphrey attend?] -> [ Illinois Institute of Technology]
Computing left vector (u)...
Selected u projection object Watts Humphrey
Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 5 | Sentence: What university did Watts Humphrey attend? Illinois Institute of | Token: rey
Rewrite layer is 5
Tying optimization objective to 27
Recording initial value of v*
loss 9.825 = 9.825 + 0.0 + 0.0 avg prob of [ Illinois Institute of Technology] 0.012164751999080181
loss 9.227 = 9.209 + 0.0 + 0.017 avg prob of [ Illinois Institute of Technology] 0.017496520653367043
loss 8.571 = 8.544 + 0.0 + 0.027 avg prob of [ Illinois Institute of Technology] 0.026704803109169006
loss 7.825 = 7.79 + 0.001 + 0.035 avg prob of [ Illinois Institute of Technology] 0.0799083262681961
loss 7.104 = 7.062 + 0.001 + 0.041 avg prob of [ Illinois Institute of Technology] 0.13679082691669464
loss 6.536 = 6.488 + 0.001 + 0.0